In [ ]:
# %% [markdown]
# # Simulation testen
#
# Dieses Notebook lädt die Simulationslogik aus `backend/app/simulation/sim.py`,
# erzeugt Beispiel-Artikel und Boxen und gibt insbesondere die Packdichte aus.

import os
import sys
from pathlib import Path

# Projektpfad so setzen, dass `app.*` importierbar ist
PROJECT_ROOT = Path.cwd().resolve()

# Falls das Notebook in /notebooks liegt, auf /backend erweitern
if (PROJECT_ROOT / "backend").exists():
    BACKEND_ROOT = PROJECT_ROOT / "backend"
elif PROJECT_ROOT.name == "notebooks" and (PROJECT_ROOT.parent / "backend").exists():
    BACKEND_ROOT = PROJECT_ROOT.parent / "backend"
else:
    BACKEND_ROOT = PROJECT_ROOT

if str(BACKEND_ROOT) not in sys.path:
    sys.path.insert(0, str(BACKEND_ROOT))

print("Backend root:", BACKEND_ROOT)

# %%
from app.simulation.sim import SimulationConfig, PackagingSimulation
from app.packing.models import Item, Box

# %% [markdown]
# ## 1. Beispiel-Daten definieren
#
# Passe hier Artikel, STL-Datei, Kollisionsdatei und Boxen an.

# %%
item = Item(
    length=40,   # mm
    width=30,    # mm
    height=20,   # mm
)

boxes = [
    Box(name="Kleine Box", length=120, width=100, height=80, capacityLHM=1),
    Box(name="Mittlere Box", length=160, width=120, height=100, capacityLHM=1),
    Box(name="Große Box", length=220, width=160, height=140, capacityLHM=1),
]

config = SimulationConfig(
    item=item,
    item_quantity=100,
    boxes=boxes,
    stl_file="ausgabe.stl",          # ggf. anpassen
    collision_file=None,             # oder z.B. "ausgabe_vhacd.obj"
    mesh_volume=0.0,                 # falls bekannt: Volumen in mm^3 eintragen
    item_mass=0.01,
    mesh_scale=(0.001, 0.001, 0.001),
    wall_thickness=0.01,
    fixed_time_step=1 / 480,
    solver_iterations=80,
    max_simulation_steps=350,
    min_simulation_steps=60,
    settle_check_interval=20,
    height_change_mm=0.5,
    settle_duration=0.25,
    settle_force_scale=0.04,
    settle_frequency=10.0,
    fit_height_tolerance=0.02,
    random_seed=42,
    runs_per_box=1,
    use_gui=False,
    parallel_simulations=False,      # fürs Notebook erstmal einfacher
)

config

# %% [markdown]
# ## 2. Simulation ausführen

# %%
simulation = PackagingSimulation(config)
result = simulation.run()
result

# %% [markdown]
# ## 3. Ergebnisse lesbar anzeigen

# %%
results = result.get("results", [])

if not results:
    print("Keine passende Box gefunden oder keine Ergebnisse vorhanden.")
else:
    for i, entry in enumerate(results, start=1):
        box = entry["box"]
        print(f"Ergebnis {i}")
        print(f"  Box: {box['name']}")
        print(f"  Maße: {box['length']} x {box['width']} x {box['height']} mm")
        print(f"  Füllhöhe: {entry['filling_height_mm']:.2f} mm")
        print(f"  Packdichte: {entry['packing_density_percent']:.2f} %")
        print(f"  Box-Auslastung: {entry['box_utilization_percent']:.2f} %")
        print(f"  Passt in Box: {entry['fits_in_box']}")
        print(f"  Artikel pro LHM: {entry.get('articles_per_lhm')}")
        print()

# %% [markdown]
# ## 4. Nur die beste Packdichte ausgeben

# %%
if results:
    best = max(results, key=lambda x: x["packing_density_percent"])
    print("Beste Packdichte:")
    print(f"  Box: {best['box']['name']}")
    print(f"  Packdichte: {best['packing_density_percent']:.2f} %")
    print(f"  Füllhöhe: {best['filling_height_mm']:.2f} mm")
else:
    print("Keine Ergebnisse verfügbar.")

# %% [markdown]
# ## 5. Optional: Ergebnisse als DataFrame

# %%
try:
    import pandas as pd

    if results:
        df = pd.DataFrame(
            [
                {
                    "box_name": r["box"]["name"],
                    "box_length_mm": r["box"]["length"],
                    "box_width_mm": r["box"]["width"],
                    "box_height_mm": r["box"]["height"],
                    "filling_height_mm": r["filling_height_mm"],
                    "packing_density_percent": r["packing_density_percent"],
                    "box_utilization_percent": r["box_utilization_percent"],
                    "fits_in_box": r["fits_in_box"],
                    "articles_per_lhm": r.get("articles_per_lhm"),
                }
                for r in results
            ]
        )
        display(df)
    else:
        print("Keine DataFrame-Daten vorhanden.")
except ImportError:
    print("pandas ist nicht installiert.")

# %% [markdown]
# ## 6. Einzelbox gezielt testen
#
# Falls du nur eine einzige Box untersuchen willst.

# %%
single_box_config = SimulationConfig(
    item=item,
    item_quantity=25,
    boxes=[boxes[0]],
    stl_file="ausgabe.stl",
    collision_file=None,
    mesh_volume=0.0,
    use_gui=False,
    parallel_simulations=False,
)

single_result = PackagingSimulation(single_box_config).run()
single_result



pybullet ok
Backend root: C:\Users\QJ095K\Desktop\BinPacking neu\test\backend
Spawne 100 Artikel...
Simuliere physikalischen Fall...

Ergebnisse
Füllhöhe in der Kiste: 61.68 mm
Relative Füllhöhe der Kiste: 77.10 %
Theoretisches Artikelvolumen gesamt: 2400.00 cm^3
Beanspruchtes Box-Volumen: 740.14 cm^3
Erreichte Packdichte: 324.26 %
Spawne 100 Artikel...
Simuliere physikalischen Fall...

Ergebnisse
Füllhöhe in der Kiste: 49.42 mm
Relative Füllhöhe der Kiste: 49.42 %
Theoretisches Artikelvolumen gesamt: 2400.00 cm^3
Beanspruchtes Box-Volumen: 948.90 cm^3
Erreichte Packdichte: 252.93 %
Spawne 100 Artikel...
Simuliere physikalischen Fall...

Ergebnisse
Füllhöhe in der Kiste: 40.04 mm
Relative Füllhöhe der Kiste: 28.60 %
Theoretisches Artikelvolumen gesamt: 2400.00 cm^3
Beanspruchtes Box-Volumen: 1409.46 cm^3
Erreichte Packdichte: 170.28 %
Ergebnis 1
  Box: Kleine Box
  Maße: 120 x 100 x 80 mm
  Füllhöhe: 61.68 mm
  Packdichte: 324.26 %
  Box-Auslastung: 250.00 %
  Passt in Box: True
  Arti

,box_name,box_length_mm,box_width_mm,box_height_mm,filling_height_mm,packing_density_percent,box_utilization_percent,fits_in_box,articles_per_lhm
0,Kleine Box,120,100,80,61.677981,324.264829,250.000000,True,100
1,Mittlere Box,160,120,100,49.421749,252.925083,125.000000,True,100
2,Große Box,220,160,140,40.041565,170.277604,48.701299,True,100


Spawne 25 Artikel...
Simuliere physikalischen Fall...

Ergebnisse
Füllhöhe in der Kiste: 30.05 mm
Relative Füllhöhe der Kiste: 37.56 %
Theoretisches Artikelvolumen gesamt: 600.00 cm^3
Beanspruchtes Box-Volumen: 360.55 cm^3
Erreichte Packdichte: 166.41 %
Spawne 25 Artikel...
Simuliere physikalischen Fall...

Ergebnisse
Füllhöhe in der Kiste: 30.05 mm
Relative Füllhöhe der Kiste: 37.56 %
Theoretisches Artikelvolumen gesamt: 600.00 cm^3
Beanspruchtes Box-Volumen: 360.55 cm^3
Erreichte Packdichte: 166.41 %

GUI aktiv -- Fenster schliessen zum Beenden.


error: Not connected to physics server.

In [6]:
gui_config = SimulationConfig(
    item=item,
    item_quantity=100,
    boxes=[boxes[0]],
    stl_file="ausgabe.stl",
    collision_file=None,
    use_gui=True,
    parallel_simulations=False,
)
gui_result = PackagingSimulation(gui_config).run()
gui_result

Spawne 100 Artikel...
Simuliere physikalischen Fall...

Ergebnisse
Füllhöhe in der Kiste: 63.84 mm
Relative Füllhöhe der Kiste: 79.80 %
Theoretisches Artikelvolumen gesamt: 2400.00 cm^3
Beanspruchtes Box-Volumen: 766.09 cm^3
Erreichte Packdichte: 313.28 %

GUI aktiv -- Fenster schliessen zum Beenden.


error: Not connected to physics server.